## Необходимые импорты

In [ ]:
!pip install catboost -q
!pip install ipywidgets -q
!pip install notebook

In [1]:
import ipywidgets as widgets
widgets.IntSlider()

IntSlider(value=0)

In [2]:
from catboost.datasets import titanic

import numpy as np

from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier, Pool, metrics, cv
from sklearn.metrics import accuracy_score

In [24]:
!pip install scikit-learn

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.1 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.1 MB 1.5 MB/s eta 0:00:06
   ----- ---------------------------------- 1.0/8.1 MB 1.5 MB/s eta 0:00:05
   ------ --------------------------------- 1.3/8.1 MB 1.5 MB/s eta 0:00:05
   ------- -------------------------------- 1.6/8.1 MB 1.5 MB/s eta 0:00:05
   ---------- ----------------------------- 2.1/8.1 MB 1.5 MB/s eta 0:00:05
   ----------- ---------------------------- 2.4/8.1 MB 1.5 MB/s eta 0:00:04
   ------------ --------------------------- 2.6/8.1 MB 1.4 MB/s eta 0:00:04
   --------------- ------------------------ 3.1/8.1 MB 1.5 MB/s eta 0:00:04
   ---------------- ----------------------- 3.4/8.1 MB 1.5 MB/s eta 0:00:04
   ---------------- ----------------------- 3.4/8.1 MB 1.5 MB/s eta 0:00:04
   ------------------- ----------

## Загрузка данных 

In [3]:
train_df, test_df = titanic()

train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Обработка данных

In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [4]:
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


In [5]:
num_value_stats = train_df.isnull().sum(axis = 0)
num_value_stats[num_value_stats != 0]

Age         177
Cabin       687
Embarked      2
dtype: int64

In [6]:
num_value_stats = test_df.isnull().sum(axis = 0)
num_value_stats[num_value_stats != 0]

Age       86
Fare       1
Cabin    327
dtype: int64

### Заполним пропуски значением -999

In [7]:
# Выбираем только те колонки, у которых тип 'string'
string_cols = train_df.select_dtypes(include='string').columns

# Применяем преобразование
train_df[string_cols] = train_df[string_cols].astype(object)
test_df[string_cols] = test_df[string_cols].astype(object)

In [8]:
train_df.fillna(-999, inplace = True)
test_df.fillna(-999, inplace = True)

### Разобьем данные на матрицы X и y

In [9]:
X = train_df.drop('Survived', axis = 1)
y = train_df.Survived

In [10]:
X.dtypes

categorical_features_indices = np.where(X.dtypes != float)[0]

In [11]:
categorical_features_indices

array([ 0,  1,  2,  3,  5,  6,  7,  9, 10])

In [12]:
X

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,-999,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,-999,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,-999,S
...,...,...,...,...,...,...,...,...,...,...,...
886,887,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,-999,S
887,888,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,-999.0,1,2,W./C. 6607,23.4500,-999,S
889,890,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


### Разобьем данные на трейн и валидацию 

In [13]:
Xtrain, Xval, ytrain, yval = train_test_split(X, y, train_size=0.75, random_state=42)

Xtest = test_df

In [14]:
model = CatBoostClassifier(
    custom_loss=[metrics.Accuracy()],
    random_seed=42
)

In [15]:
model.fit(
    Xtrain, ytrain,
    cat_features=categorical_features_indices,
    eval_set=(Xval, yval),
    logging_level='Silent', # хотим ли мы видеть метрики на каждой итерации
    plot = True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

CatBoostClassifier(custom_loss=['Accuracy'], random_seed=42)

### Оценим качество по кросс-валидации

In [39]:
cv_params = model.get_params()
cv_params.update({
    'loss_function': metrics.Logloss()
})
cv_data = cv(
    Pool(X, y, cat_features=categorical_features_indices),
    cv_params,
    logging_level='Silent',
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [40]:
print('Best validation accuracy score: {:.2f}±{:.2f} on step {}'.format(
    np.max(cv_data['test-Accuracy-mean']),
    cv_data['test-Accuracy-std'][np.argmax(cv_data['test-Accuracy-mean'])],
    np.argmax(cv_data['test-Accuracy-mean'])
))

Best validation accuracy score: 0.83±0.02 on step 355


### Применяем обученную модель

In [42]:
pred = model.predict(Xtest)
pred_prob = model.predict_proba(Xtest)
print(pred[:10])
print(pred_prob[:10])

[0 0 0 0 1 0 1 0 1 0]
[[0.85473931 0.14526069]
 [0.76313031 0.23686969]
 [0.88972889 0.11027111]
 [0.87876173 0.12123827]
 [0.3611047  0.6388953 ]
 [0.90513381 0.09486619]
 [0.33434185 0.66565815]
 [0.78468564 0.21531436]
 [0.39429048 0.60570952]
 [0.94047549 0.05952451]]


### Улучшение предсказаний и др. возможности CatBoost

#### Early Stopping

In [46]:
model.fit(
    Xtrain, ytrain,
    cat_features=categorical_features_indices,
    eval_set=(Xval, yval),
    early_stopping_rounds=30, # если начало ухудшаться кач-во, модель обучается еще 30 деревьев. По умолчанию 1000
    logging_level='Verbose', # если хотим видеть метрики на каждой итерации
    plot = True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Learning rate set to 0.028683
0:	learn: 0.6739988	test: 0.6742630	best: 0.6742630 (0)	total: 25.3ms	remaining: 25.3s
1:	learn: 0.6589013	test: 0.6592240	best: 0.6592240 (1)	total: 39.3ms	remaining: 19.6s
2:	learn: 0.6421502	test: 0.6426778	best: 0.6426778 (2)	total: 63.3ms	remaining: 21s
3:	learn: 0.6297276	test: 0.6302310	best: 0.6302310 (3)	total: 83.7ms	remaining: 20.8s
4:	learn: 0.6147184	test: 0.6198228	best: 0.6198228 (4)	total: 110ms	remaining: 21.9s
5:	learn: 0.6017730	test: 0.6073627	best: 0.6073627 (5)	total: 133ms	remaining: 22s
6:	learn: 0.5885309	test: 0.5956000	best: 0.5956000 (6)	total: 154ms	remaining: 21.9s
7:	learn: 0.5783200	test: 0.5858523	best: 0.5858523 (7)	total: 178ms	remaining: 22s
8:	learn: 0.5665895	test: 0.5743842	best: 0.5743842 (8)	total: 203ms	remaining: 22.4s
9:	learn: 0.5575381	test: 0.5662283	best: 0.5662283 (9)	total: 227ms	remaining: 22.5s
10:	learn: 0.5491045	test: 0.5575176	best: 0.5575176 (10)	total: 249ms	remaining: 22.4s
11:	learn: 0.5423887	tes

CatBoostClassifier(custom_loss=['Accuracy'], random_seed=42)

In [44]:
model.tree_count_

284

#### Важность признаков 

CatBoost поддерживает несколько способов вычисления важности признаков, в том числе широко применяемый сейчас подход Shap (про него поговорим в следующих модулях).

In [48]:
feature_importances = model.get_feature_importance()

feature_names = Xtrain.columns
for score, name in sorted(zip(feature_importances, feature_names), reverse=True):
    print('{}: {}'.format(name, score))

Sex: 28.377591527551807
Pclass: 17.450379813673287
Parch: 10.276200044515498
Embarked: 8.761954037905873
Cabin: 8.281577549519369
SibSp: 7.950157281933983
Age: 7.842375602284014
Ticket: 5.620556803330715
Fare: 5.4392073392855105
PassengerId: 0.0
Name: 0.0


#### Сохранение модели

In [49]:
# сохраняем модель
model.save_model('catboost_model.dump')

# загружаем сохраненную модель
model.load_model('catboost_model.dump');